In [ ]:
# === 1. Import thư viện ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

# === 2. Đọc dữ liệu CSV đã gắn nhãn ===
df = pd.read_csv("/content/sample_data/all_labeled_questions.csv")  # 🔹 đường dẫn tới file bạn tải lên

# Kiểm tra dữ liệu
print(df.head())

# === 3. Ánh xạ nhãn thành số ===
label_map = {"question": 0, "answer": 1}
df["label_id"] = df["label"].map(label_map)

# === 4. Chia tập train/test ===
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

# === 5. Chuẩn bị tokenizer ===
model_name = "vinai/phobert-base"  # ⚠️ Nếu dữ liệu là tiếng Việt, nên đổi thành "bert-base-multilingual-cased" hoặc "PhoBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

# === 6. Tạo dataset cho HuggingFace ===
train_dataset = Dataset.from_pandas(train_df[['content', 'label_id']])
test_dataset = Dataset.from_pandas(test_df[['content', 'label_id']])

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label_id", "labels")
test_dataset = test_dataset.rename_column("label_id", "labels")

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# === 7. Load mô hình gốc ===
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_map))

# === 8. Cấu hình huấn luyện ===
training_args = TrainingArguments(
    output_dir="./model_output",
    do_eval=True,   # để mô hình vẫn chạy đánh giá trên eval_dataset
    save_total_limit=2,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=50,
    fp16=torch.cuda.is_available(),  # bật FP16 nếu có GPU
    report_to="none"
)

# === 9. Huấn luyện ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# === 10. Lưu mô hình đã fine-tune ===
save_path = "/content/bert_question_answer_classifier"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Fine-tune hoàn tất. Mô hình đã lưu tại:", save_path)
print("Label map:", label_map)


In [ ]:
import os
from transformers import pipeline
from docx import Document
import pandas as pd
import re

# === 1. Load mô hình PhoBERT đã fine-tune ===
model_path = "/content/bert_question_answer_classifier"
clf = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,  # ✅ dùng tokenizer của PhoBERT fine-tune
    return_all_scores=False,
    device=-1  # GPU = 0, CPU = -1
)

# === 2. Hàm đọc đoạn văn từ DOCX ===
def extract_paragraphs_from_docx(docx_path):
    doc = Document(docx_path)
    return [p.text.strip() for p in doc.paragraphs if p.text.strip()]

# === 3. Hàm xử lý từng file ===
def classify_docx(docx_path):
    paragraphs = extract_paragraphs_from_docx(docx_path)
    label_mapping = {"LABEL_0": "question", "LABEL_1": "answer"}

    classified = []
    for text in paragraphs:
        try:
            pred = clf(text)[0]
            label = label_mapping.get(pred["label"], pred["label"])
            classified.append({"text": text, "label": label})
        except Exception as e:
            classified.append({"text": text, "label": "error"})
            print(f"Lỗi khi xử lý: {text[:40]} -> {e}")

    # Gom nhóm question → answer
    grouped = []
    current_question = None
    current_answers = []

    for item in classified:
        text = item["text"]
        label = item["label"]

        if label == "question":
            if current_question and current_answers:
                grouped.append({
                    "question": current_question,
                    "answer": current_answers
                })
            current_question = text
            current_answers = []

        elif label == "answer":
            if not re.match(r"^[A-D][\.\)]", text):
                prefix = chr(65 + len(current_answers)) + ". "
                text = prefix + text
            current_answers.append(text)

    if current_question and current_answers:
        grouped.append({
            "question": current_question,
            "answer": current_answers
        })

    # Trả về DataFrame
    if grouped:
        df = pd.DataFrame(grouped)
        df["answer"] = df["answer"].apply(lambda x: str(x))
        return df
    else:
        print(f"⚠️ Không tìm thấy question/answer trong file: {os.path.basename(docx_path)}")
        return pd.DataFrame(columns=["question", "answer"])

# === 4. Xử lý nhiều file DOCX và gộp thành 1 CSV ===
input_folder = "/content/docx/"  # 📂 thư mục chứa các file .docx của bạn
output_csv = "/content/sample_data/all_questions_combined.csv"

# lấy tất cả file .docx
docx_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".docx")]
print(f"📄 Tìm thấy {len(docx_files)} file DOCX.")

# xử lý từng file
all_dataframes = []
for path in docx_files:
    print(f"🔹 Đang xử lý: {os.path.basename(path)}")
    df = classify_docx(path)
    if not df.empty:
        df["source_file"] = os.path.basename(path)
        all_dataframes.append(df)

# gộp tất cả
if all_dataframes:
    final_df = pd.concat(all_dataframes, ignore_index=True)
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"✅ Hoàn tất! Tổng {len(final_df)} câu hỏi. File lưu tại: {output_csv}")
else:
    print("⚠️ Không có dữ liệu hợp lệ nào được xuất.")
